In [1]:
import pandas as pd
from sentence_transformers import SentenceTransformer
import chromadb
import json

# Initialize the Local Setup
# This creates a folder named 'pels_vector_db' in your current directory.
chroma_client = chromadb.PersistentClient(path="./pels_vector_db")

# Create a collection (think of it like an SQL table)
collection = chroma_client.get_or_create_collection(name="prompt_examples")

# Load a fast, lightweight local transformer model. 
# It will download once (~90MB) and then run entirely locally.
print("Loading local embedding model...")
model = SentenceTransformer('all-MiniLM-L6-v2') 
print("Model loaded!")

Loading local embedding model...


Loading weights:   0%|          | 0/103 [00:00<?, ?it/s]

BertModel LOAD REPORT from: sentence-transformers/all-MiniLM-L6-v2
Key                     | Status     |  | 
------------------------+------------+--+-
embeddings.position_ids | UNEXPECTED |  | 

Notes:
- UNEXPECTED:	can be ignored when loading from different task/architecture; not ok if you expect identical arch.


Model loaded!


In [2]:
print("Loading Excel file...")
# Using your specific file path
df = pd.read_excel(r"D:/PROJECTS/PELS2/Database/Trinetri Datsets/PELS_CAREER_v4,7.4+_fixed.xlsx")

# Clean up empty rows/NaNs
df = df.dropna(how='all')

documents = []
metadatas = []
ids = []

print("Formatting rows for embedding...")
for index, row in df.iterrows():
    # Drop empty columns for this specific row to keep data clean
    row_dict = row.dropna().to_dict()
    
    # THE TRICK: Convert the row dictionary into a JSON string.
    content_string = json.dumps(row_dict)
    documents.append(content_string)
    
    # Metadata lets you hard-filter later (e.g., "Only search within the 'CODE' domain")
    metadata = {
        "domain": str(row_dict.get("domain", "UNKNOWN")),
        "skill_level": str(row_dict.get("skill_level", "UNKNOWN"))
    }
    metadatas.append(metadata)
    
    # Give each row a unique ID
    ids.append(f"prompt_{index}")

print(f"Successfully processed {len(documents)} rows from Excel.")

Loading Excel file...
Formatting rows for embedding...
Successfully processed 600 rows from Excel.


In [3]:
print(f"Generating embeddings for {len(documents)} rows (This might take a minute)...")
embeddings = model.encode(documents).tolist()

print("Saving to local Vector DB...")
collection.add(
    embeddings=embeddings,
    documents=documents,
    metadatas=metadatas,
    ids=ids
)

print("Success! Your local vector database is ready.")

Generating embeddings for 600 rows (This might take a minute)...
Saving to local Vector DB...
Success! Your local vector database is ready.


In [4]:
import json

# Define a fake student prompt to test the search
test_student_prompt = "I am designing a 2-day career development workshop for 30 mid-career professionals (7-15 years experience) at a pharmaceutical company. The cohort is diverse in function (sales, regulatory, medical affairs, manufacturing) and career stage. The company's goal: reduce mid-career attrition by improving internal mobility awareness and career ownership. My challenge: these professionals are skeptical — many have been to generic HR workshops before. Task 1: Design the 2-day workshop arc (morning and afternoon sessions per day) with a brief rationale for each block. Task 2: Identify 3 design principles that will make this workshop feel substantively different from generic HR workshops. Task 3: Design 1 workshop activity for Day 1 that forces participants to produce a real career artifact they can use immediately after the workshop ends. Task 4: Flag 2 assumptions I am making about this cohort that I should validate before designing further. Task 5: Tell me what this workshop design cannot do and what ongoing support structure would be needed to make the attrition impact real."


print(f"Searching for prompts similar to: '{test_student_prompt}'\n")

# 1. Embed the test prompt
query_embedding = model.encode([test_student_prompt]).tolist()

# 2. Query the database
results = collection.query(
    query_embeddings=query_embedding,
    n_results=2,
    where={"domain": "CAREER"} 
)

# 3. Display the results clearly
print("--- TOP MATCHES FOUND ---\n")
for idx, document_string in enumerate(results['documents'][0]):
    doc_data = json.loads(document_string)
    
    print(f"Match {idx + 1}:")
    print(f"Prompt ID: {doc_data.get('prompt_id', 'N/A')}")
    print(f"Skill Level: {doc_data.get('skill_level', 'N/A')}")
    print(f"Final Score: {doc_data.get('final_score', 'N/A')}")
    
    # Grab the individual column scores
    c1 = doc_data.get('c1_score', 'N/A')
    c2 = doc_data.get('c2_score', 'N/A')
    c3 = doc_data.get('c3_score', 'N/A')
    c4 = doc_data.get('c4_score', 'N/A')
    c5 = doc_data.get('c5_score', 'N/A')
    c6 = doc_data.get('c6_score', 'N/A')
    
    print(f"Scores Breakdown: C1:{c1} | C2:{c2} | C3:{c3} | C4:{c4} | C5:{c5} | C6:{c6}")
    
    # Print the actual text and justification
    print(f"\nPrompt Text:\n{doc_data.get('prompt_text', 'N/A')}")
    print(f"\nJustification:\n{doc_data.get('justification', 'N/A')}")
    print("-" * 60 + "\n")

Searching for prompts similar to: 'I am designing a 2-day career development workshop for 30 mid-career professionals (7-15 years experience) at a pharmaceutical company. The cohort is diverse in function (sales, regulatory, medical affairs, manufacturing) and career stage. The company's goal: reduce mid-career attrition by improving internal mobility awareness and career ownership. My challenge: these professionals are skeptical — many have been to generic HR workshops before. Task 1: Design the 2-day workshop arc (morning and afternoon sessions per day) with a brief rationale for each block. Task 2: Identify 3 design principles that will make this workshop feel substantively different from generic HR workshops. Task 3: Design 1 workshop activity for Day 1 that forces participants to produce a real career artifact they can use immediately after the workshop ends. Task 4: Flag 2 assumptions I am making about this cohort that I should validate before designing further. Task 5: Tell me w

In [5]:
import pandas as pd
from sentence_transformers import SentenceTransformer
import chromadb
import json
import os
import uuid

# 1. Connect to your EXISTING local database
print("Connecting to existing database...")
chroma_client = chromadb.PersistentClient(path="./pels_vector_db")
collection = chroma_client.get_collection(name="prompt_examples")

# 2. Load the model
print("Loading model...")
model = SentenceTransformer('all-MiniLM-L6-v2')

# 3. List your new files here
new_files = [
    r'D:/PROJECTS/PELS2/Database/Trinetri Datsets/PELS_final_dataset.xlsx',
    r'D:/PROJECTS/PELS2/Database/Trinetri Datsets/PELS_Enhanced_Analysis.xlsx',
    r'D:/PROJECTS/PELS2/Database/Trinetri Datsets/PELS_Education_Dataset.csv'

]

# 4. Process each file
for file_path in new_files:
    file_name = os.path.basename(file_path)
    file_prefix = os.path.splitext(file_name)[0]

    print(f"\n--- Processing {file_name} ---")
    
    # Read the excel file
    try:
        df = pd.read_excel(file_path)
        df = df.dropna(how='all')
    except Exception as e:
        print(f"Error reading {file_path}: {e}")
        continue

    documents = []
    metadatas = []
    ids = []
    
    for index, row in df.iterrows():
        row_dict = row.dropna().to_dict()
        
        # Convert row to JSON string
        content_string = json.dumps(row_dict)
        documents.append(content_string)
        
        # Metadata
        metadata = {
            "domain": str(row_dict.get("domain", "UNKNOWN")),
            "skill_level": str(row_dict.get("skill_level", "UNKNOWN"))
        }
        metadatas.append(metadata)
        
        # UNIQUE ID (prevents duplicate errors on reruns)
        unique_id = f"{file_prefix}_row_{index}_{uuid.uuid4().hex[:8]}"
        ids.append(unique_id)

    if not documents:
        print("No valid rows found in this file. Skipping.")
        continue

    # Generate embeddings
    print(f"Generating embeddings for {len(documents)} rows...")
    embeddings = model.encode(documents).tolist()

    # Add to DB
    print("Adding to database...")
    collection.add(
        embeddings=embeddings,
        documents=documents,
        metadatas=metadatas,
        ids=ids
    )

    print(f"Successfully added {len(documents)} rows from {file_name}!")

# Final count
total_items = collection.count()
print(f"\n✅ All files processed! Your Vector DB now contains {total_items} prompts.")

Connecting to existing database...
Loading model...


Loading weights:   0%|          | 0/103 [00:00<?, ?it/s]

BertModel LOAD REPORT from: sentence-transformers/all-MiniLM-L6-v2
Key                     | Status     |  | 
------------------------+------------+--+-
embeddings.position_ids | UNEXPECTED |  | 

Notes:
- UNEXPECTED:	can be ignored when loading from different task/architecture; not ok if you expect identical arch.



--- Processing PELS_final_dataset.xlsx ---
Generating embeddings for 360 rows...
Adding to database...
Successfully added 360 rows from PELS_final_dataset.xlsx!

--- Processing PELS_Enhanced_Analysis.xlsx ---
Generating embeddings for 264 rows...
Adding to database...
Successfully added 264 rows from PELS_Enhanced_Analysis.xlsx!

--- Processing PELS_Education_Dataset.csv ---
Error reading D:/PROJECTS/PELS2/Database/Trinetri Datsets/PELS_Education_Dataset.csv: Excel file format cannot be determined, you must specify an engine manually.

✅ All files processed! Your Vector DB now contains 1224 prompts.


In [2]:
import pandas as pd
from sentence_transformers import SentenceTransformer
import chromadb
import json
import os
import uuid

# 1. Connect to your EXISTING local database
print("Connecting to existing database...")
chroma_client = chromadb.PersistentClient(path="./pels_vector_db")
collection = chroma_client.get_collection(name="prompt_examples")

# 2. Load the model
print("Loading model...")
model = SentenceTransformer('all-MiniLM-L6-v2')

# 3. List your new files here
new_files = [
    r'D:/PROJECTS/PELS2/Database/Trinetri Datsets/PELS_CREATE_300_curated.xlsx'


]

# 4. Process each file
for file_path in new_files:
    file_name = os.path.basename(file_path)
    file_prefix = os.path.splitext(file_name)[0]

    print(f"\n--- Processing {file_name} ---")
    
    # Read the excel file
    try:
        df = pd.read_excel(file_path)
        df = df.dropna(how='all')
    except Exception as e:
        print(f"Error reading {file_path}: {e}")
        continue

    documents = []
    metadatas = []
    ids = []
    
    for index, row in df.iterrows():
        row_dict = row.dropna().to_dict()
        
        # Convert row to JSON string
        content_string = json.dumps(row_dict)
        documents.append(content_string)
        
        # Metadata
        metadata = {
            "domain": str(row_dict.get("domain", "UNKNOWN")),
            "skill_level": str(row_dict.get("skill_level", "UNKNOWN"))
        }
        metadatas.append(metadata)
        
        # UNIQUE ID (prevents duplicate errors on reruns)
        unique_id = f"{file_prefix}_row_{index}_{uuid.uuid4().hex[:8]}"
        ids.append(unique_id)

    if not documents:
        print("No valid rows found in this file. Skipping.")
        continue

    # Generate embeddings
    print(f"Generating embeddings for {len(documents)} rows...")
    embeddings = model.encode(documents).tolist()

    # Add to DB
    print("Adding to database...")
    collection.add(
        embeddings=embeddings,
        documents=documents,
        metadatas=metadatas,
        ids=ids
    )

    print(f"Successfully added {len(documents)} rows from {file_name}!")

# Final count
total_items = collection.count()
print(f"\n✅ All files processed! Your Vector DB now contains {total_items} prompts.")

Connecting to existing database...
Loading model...


Loading weights:   0%|          | 0/103 [00:00<?, ?it/s]

BertModel LOAD REPORT from: sentence-transformers/all-MiniLM-L6-v2
Key                     | Status     |  | 
------------------------+------------+--+-
embeddings.position_ids | UNEXPECTED |  | 

Notes:
- UNEXPECTED:	can be ignored when loading from different task/architecture; not ok if you expect identical arch.



--- Processing PELS_CREATE_300_curated.xlsx ---
Generating embeddings for 300 rows...
Adding to database...
Successfully added 300 rows from PELS_CREATE_300_curated.xlsx!

✅ All files processed! Your Vector DB now contains 1524 prompts.
